# IT9002 Lab 2.1: Bag of Words and TF-IDF

**Unit 02: Text Representation**

**IT9002 Natural Language Processing**

## Objectives

By the end of this lab, the following should be clear:

- Building a document-term matrix by hand and with `CountVectorizer`, and reading its rows and columns
- Computing term frequency, document frequency, inverse document frequency and TF-IDF for a small corpus, and why the scikit-learn formula differs from the textbook one
- Representing a new document with a vectorizer fitted on other documents, and what happens to the words it has never seen
- How preprocessing settings and n-gram features change the vocabulary and the size of the matrix on a corpus of 2,225 articles
- Reading the highest-weighted terms of a document as its topic

## Data

- **BBC News dataset** (Greene and Cunningham, 2006), 2,225 articles in 5 categories, raw full-text archive, http://mlg.ucd.ie/datasets/bbc.html, provided for non-commercial and research use.

- The course file `bbc_news.csv` (4.8 MB, on Moodle).

The file `bbc_news.csv` can be downloaded from Moodle. Download it to your computer, then run the cell below. If the file is already available in the runtime, the cell loads it directly; otherwise, it opens a file-upload dialog.

`pd.read_csv()` reads the file into a **DataFrame**, and `head()` displays its first 5 rows.

In [1]:
from pathlib import Path
import pandas as pd

DATA = "bbc_news.csv"
path = next((p for p in (Path(DATA), Path("data") / DATA) if p.exists()), None)
if path is None:
    from google.colab import files
    files.upload()
    path = Path(DATA)

df = pd.read_csv(path)
print(df.shape)
df.head()

(2225, 3)


,category,title,text
0,business,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarner...
1,business,Dollar gains on Greenspan speech,The dollar has hit its highest level against t...
2,business,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuko...
3,business,High fuel prices hit BA's profits,British Airways has blamed high fuel prices fo...
4,business,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Domec...


The file holds 2,225 articles, each with a category, a title and the article text.

## Setup

scikit-learn, NumPy and pandas are preinstalled in Google Colab, so the cell below does not install them there. In other environments, scikit-learn will be installed if it is not already available.

In [2]:
import importlib.util, subprocess, sys

if importlib.util.find_spec("sklearn") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"], check=True)

The cell below imports the tools used in this lab and prints the versions of Python, pandas, NumPy and scikit-learn.

In [3]:
import platform
import numpy as np
import sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

print("Python", platform.python_version())
print("pandas", pd.__version__, "| NumPy", np.__version__, "| scikit-learn", sklearn.__version__)

Python 3.11.9
pandas 3.0.2 | NumPy 2.4.2 | scikit-learn 1.9.1


## 1. Three Documents as Vectors

**Text representation** is the process of converting a document into a numerical form that a model can process. In the **bag-of-words** representation, each document becomes a vector with one position per vocabulary term, and the position holds the number of times the term occurs in the document.

Three short documents are used first, so that every number can be checked by hand. Each document is 5 words long, and together they contain 8 distinct terms.

The cell below defines the documents and builds the vocabulary: the sorted set of the distinct words.

In [4]:
docs = ["shares rose on strong profits",
        "shares fell on weak profits",
        "profits fell and shares fell"]

vocab = sorted({word for doc in docs for word in doc.split()})
print(len(vocab), "terms:", vocab)

8 terms: ['and', 'fell', 'on', 'profits', 'rose', 'shares', 'strong', 'weak']


The vocabulary contains **8 terms** in alphabetical order. Each term is one column of the matrix.

The cell below counts, for each document, how many times each vocabulary term occurs, and displays the result as a **document-term matrix**: one row per document, one column per term.

In [5]:
counts = np.array([[doc.split().count(term) for term in vocab] for doc in docs])
C = pd.DataFrame(counts, index=["d1", "d2", "d3"], columns=vocab)
C

,and,fell,on,profits,rose,shares,strong,weak
d1,0,0,1,1,1,1,1,0
d2,0,1,1,1,0,1,0,1
d3,1,2,0,1,0,1,0,0


The term `fell` occurs **twice** in d3, so its cell holds 2. A term that does not occur in a document holds 0. The order of the words is lost: the row of d3 would be the same for `fell shares and fell profits`.

`CountVectorizer` builds the same matrix. `fit_transform()` learns the vocabulary from the documents and converts them into rows of counts in one call. `get_feature_names_out()` returns the column names.

The cell below builds the matrix with `CountVectorizer` and compares it with the hand-built one.

In [6]:
cv = CountVectorizer()
X = cv.fit_transform(docs)
print(cv.get_feature_names_out())
print(X.toarray())
print("same as the hand-built matrix:", np.array_equal(X.toarray(), counts))

['and' 'fell' 'on' 'profits' 'rose' 'shares' 'strong' 'weak']
[[0 0 1 1 1 1 1 0]
 [0 1 1 1 0 1 0 1]
 [1 2 0 1 0 1 0 0]]
same as the hand-built matrix: True


The two matrices are **identical**. `CountVectorizer` lowercases the text and keeps tokens of two or more word characters by default, which changes nothing on these three documents.

`X` is a **sparse matrix**, which stores only the non-zero cells. `toarray()` converts it to an ordinary NumPy array for reading, which is fine for 3 rows and 8 columns and is avoided on a large matrix.

## 2. Term Frequency, Document Frequency and IDF

**Term frequency (TF)** is the number of times a term occurs in one document, which is the value already in the matrix.

**Document frequency (DF)** is the number of documents that contain a term. It is counted down each column: the number of cells that are greater than 0.

The cell below computes the DF of every term.

In [7]:
DF = (C > 0).sum()
DF

and        1
fell       2
on         2
profits    3
rose       1
shares     3
strong     1
weak       1
dtype: int64

`profits` and `shares` occur in **all 3 documents**, `fell` and `on` in 2, and the other four terms in 1 document each.

**Inverse document frequency (IDF)** turns DF into a weight that is high for rare terms and low for common ones. Two formulas are in use:

- **First:** the textbook formula, ln(N / DF), where N is the number of documents.

- **Second:** the scikit-learn formula, ln((1 + N) / (1 + DF)) + 1, which adds 1 inside the logarithm, above and below the line, and 1 to the result.

The cell below computes both for the 8 terms.

In [8]:
N = len(docs)
idf_textbook = np.log(N / DF)
idf_sklearn = np.log((1 + N) / (1 + DF)) + 1
pd.DataFrame({"DF": DF, "IDF textbook": idf_textbook.round(3), "IDF scikit-learn": idf_sklearn.round(3)})

,DF,IDF textbook,IDF scikit-learn
and,1,1.099,1.693
fell,2,0.405,1.288
on,2,0.405,1.288
profits,3,0.000,1.000
rose,1,1.099,1.693
shares,3,0.000,1.000
strong,1,1.099,1.693
weak,1,1.099,1.693


Under the textbook formula, a term that occurs in every document receives an IDF of **0**, so it would disappear from every vector. Under the scikit-learn formula the same term receives **1.000**, and a term in one document receives **1.693** in place of 1.099.

The added 1s are a form of **smoothing**: the 1s inside the logarithm behave as if one extra document contained every term, and the 1 outside keeps every term at a weight of at least 1.

## 3. Computing TF-IDF

**TF-IDF** multiplies the term frequency in a document by the inverse document frequency of the term across the corpus. A term scores highly when it is frequent in this document and rare in the others.

The cell below multiplies every cell of the count matrix by the IDF of its column.

In [9]:
W = C * idf_sklearn
W.round(3)

,and,fell,on,profits,rose,shares,strong,weak
d1,0.000,0.000,1.288,1.0,1.693,1.0,1.693,0.000
d2,0.000,1.288,1.288,1.0,0.000,1.0,0.000,1.693
d3,1.693,2.575,0.000,1.0,0.000,1.0,0.000,0.000


`fell` in d3 scores 2 × 1.288 = **2.575**, the highest value in the table. `profits` and `shares` score 1.000 in every document, because they occur everywhere.

`TfidfVectorizer` then divides every row by its **Euclidean length**, the square root of the sum of its squared values, so that each row has length 1. This places long and short documents on the same scale.

The cell below computes the length of each row and divides by it.

In [10]:
lengths = np.sqrt((W ** 2).sum(axis=1))
print(lengths.round(3).to_dict())
T_hand = W.div(lengths, axis=0)
T_hand.round(3)

{'d1': 3.065, 'd2': 2.861, 'd3': 3.391}


,and,fell,on,profits,rose,shares,strong,weak
d1,0.000,0.000,0.42,0.326,0.552,0.326,0.552,0.000
d2,0.000,0.450,0.45,0.350,0.000,0.350,0.000,0.592
d3,0.499,0.759,0.00,0.295,0.000,0.295,0.000,0.000


The length of d1 is **3.065**, so `rose` in d1 becomes 1.693 / 3.065 = 0.552. After the division, the squares in each row sum to 1.

The cell below builds the matrix with `TfidfVectorizer` and checks that it matches the hand computation. `np.allclose()` returns `True` when every pair of values agrees to within a small tolerance.

In [11]:
tv = TfidfVectorizer()
T = tv.fit_transform(docs)
print("IDF learned by the library:", tv.idf_.round(3))
print("matches the hand computation:", np.allclose(T.toarray(), T_hand.values))

IDF learned by the library: [1.693 1.288 1.288 1.    1.693 1.    1.693 1.693]
matches the hand computation: True


The library's IDF values are the scikit-learn column of the table in section 2, and its matrix matches the hand computation.

The cell below repeats the check with the textbook IDF.

In [12]:
W_textbook = C * idf_textbook
T_textbook = W_textbook.div(np.sqrt((W_textbook ** 2).sum(axis=1)), axis=0)
print("textbook version matches the library:", np.allclose(T.toarray(), T_textbook.values))
print(T_textbook.round(3))

textbook version matches the library: False
      and   fell     on  profits   rose  shares  strong   weak
d1  0.000  0.000  0.253      0.0  0.684     0.0   0.684  0.000
d2  0.000  0.327  0.327      0.0  0.000     0.0   0.000  0.887
d3  0.805  0.594  0.000      0.0  0.000     0.0   0.000  0.000


The textbook version does **not** match. Its `profits` and `shares` columns are 0 in every row, so the two versions of the same document differ in every position. A hand computation is compared with the library only when it uses the library's formula.

## 4. Transforming a New Document

A vectorizer is **fitted** on one set of documents and then **transforms** other documents. Fitting fixes the vocabulary and the IDF values. Transforming converts any document into a vector over that fixed vocabulary, so a word the vectorizer has never seen has no column and is dropped.

A model trained on the vectors expects the same columns in the same order. For this reason, the vectorizer is fitted on the training documents only, and a new document is transformed, never used for fitting.

The cell below transforms a new document with `tv`, the vectorizer fitted on the three documents in section 3, and lists the words of the new document that have no column.

In [13]:
new_doc = "shares rose on record profits and strong sales"
row = tv.transform([new_doc])
print(dict(zip(tv.get_feature_names_out(), row.toarray()[0].round(3).tolist())))
print("dropped:", [word for word in new_doc.split() if word not in tv.vocabulary_])

{'and': 0.484, 'fell': 0.0, 'on': 0.368, 'profits': 0.286, 'rose': 0.484, 'shares': 0.286, 'strong': 0.484, 'weak': 0.0}
dropped: ['record', 'sales']


`record` and `sales` are **dropped**, and the vector is built from the other 6 words alone. The weights come from the IDF values learned on the three documents, and the row is normalized over the words that remain.

The cell below refits a vectorizer on the three documents together with the new one, to show what fitting on the new document would change.

In [14]:
tv_refit = TfidfVectorizer().fit(docs + [new_doc])
print(len(tv_refit.vocabulary_), "columns:", tv_refit.get_feature_names_out())
print("IDF:", tv_refit.idf_.round(3))

10 columns: ['and' 'fell' 'on' 'profits' 'record' 'rose' 'sales' 'shares' 'strong'
 'weak']
IDF: [1.511 1.511 1.223 1.    1.916 1.511 1.916 1.    1.511 1.916]


Refitting gives **10 columns** in place of 8 and different IDF values, so the vectors already built for d1, d2 and d3 would no longer line up with the new ones.

## 5. The Document-Term Matrix of a Corpus

The same operations scale to the 2,225 BBC articles. The cell below counts the articles in each category and the length of each article in words, split on whitespace.

In [15]:
words = df["text"].str.split().str.len()
print(f"median {words.median():.0f} words, shortest {words.min()}, longest {words.max():,}")
df.assign(words=words).groupby("category")["words"].agg(["count", "median"])

median 326 words, shortest 84, longest 4,428


,count,median
category,,
business,510,292.0
entertainment,386,256.5
politics,417,434.0
sport,511,282.0
tech,401,441.0


The articles have a median length of **326 words** and range from 84 to 4,428 words. Business and sport are the largest categories, with 510 and 511 articles, and politics and tech have the longest articles, with medians of 434 and 441 words.

The cell below builds the count matrix of the whole corpus with the default settings and prints:

- The shape of the matrix.

- The number of non-zero cells and the share of all cells they represent.

- The memory used by the sparse matrix and the memory a dense array of the same shape would use.

In [16]:
cv = CountVectorizer()
X = cv.fit_transform(df["text"])
cells = X.shape[0] * X.shape[1]
print("shape:", X.shape, f"| cells: {cells:,}")
print(f"non-zero cells: {X.nnz:,} ({X.nnz / cells:.2%} of the cells, {X.nnz / X.shape[0]:.0f} per article)")
sparse_bytes = X.data.nbytes + X.indices.nbytes + X.indptr.nbytes
print(f"sparse matrix: {sparse_bytes / 1e6:.1f} MB | dense array of the same shape: {cells * 8 / 1e6:.0f} MB")

shape: (2225, 29275) | cells: 65,136,875
non-zero cells: 445,870 (0.68% of the cells, 200 per article)
sparse matrix: 5.4 MB | dense array of the same shape: 521 MB


The matrix has **2,225 rows and 29,275 columns**, and only **0.68%** of its 65 million cells are non-zero: an article contains about 200 distinct terms out of a vocabulary of 29,275. The sparse form uses **5.4 MB**, and a dense array of the same shape would use 521 MB.

The number of columns depends on the preprocessing applied before counting. The cell below builds the matrix under five settings and records the vocabulary size and the number of tokens counted:

- `lowercase=False` keeps the original letter case.

- `stop_words="english"` removes the 318 words on the scikit-learn English stop list.

- `min_df=2` and `min_df=5` keep only the terms that occur in at least 2 or 5 articles.

In [17]:
settings = {
    "original case, every token": CountVectorizer(lowercase=False),
    "lowercased": CountVectorizer(),
    "stop words removed": CountVectorizer(stop_words="english"),
    "stop words removed, min_df=2": CountVectorizer(stop_words="english", min_df=2),
    "stop words removed, min_df=5": CountVectorizer(stop_words="english", min_df=5),
}
rows = []
for name, vec in settings.items():
    M = vec.fit_transform(df["text"])
    rows.append({"setting": name, "vocabulary": M.shape[1], "tokens counted": int(M.sum())})
pd.DataFrame(rows).set_index("setting")

,vocabulary,tokens counted
setting,,
"original case, every token",33322,824697
lowercased,29275,824697
stop words removed,28980,447008
"stop words removed, min_df=2",17351,432143
"stop words removed, min_df=5",9052,404090


Case folding removes **4,047 columns**. Stop-word removal takes away almost 1/2 of the tokens but only **295 columns**, because the stop list is short and its words are frequent. `min_df=2` removes 11,629 columns, **40%** of the vocabulary, while keeping 97% of the tokens, because those columns belong to terms that occur in a single article.

## 6. Weighting the Terms of an Article

The first article, `Ad sales boost Time Warner profit`, is used to compare the three term frequency variants:

- **Raw TF** is the number of times the term occurs.

- **Logarithmic TF** is 1 + ln(tf), which reduces the effect of repeated occurrences.

- **Normalized TF** divides the count by the length of the article in words.

The cell below counts the terms of the first article and computes the three variants for four of them.

In [18]:
cv_article = CountVectorizer(stop_words="english")
counts_article = cv_article.fit_transform([df.loc[0, "text"]]).toarray()[0]
length = len(df.loc[0, "text"].split())
raw = pd.Series({t: int(counts_article[cv_article.vocabulary_[t]]) for t in ["timewarner", "profits", "google", "quarterly"]})
print("article length:", length, "words")
pd.DataFrame({"raw TF": raw, "logarithmic TF": (1 + np.log(raw)).round(2), "normalized TF": (raw / length).round(3)})

article length: 415 words


,raw TF,logarithmic TF,normalized TF
timewarner,7,2.95,0.017
profits,5,2.61,0.012
google,2,1.69,0.005
quarterly,1,1.00,0.002


The article is **415 words** long. Raw TF lets `timewarner`, with 7 occurrences, count 7 times as much as `quarterly`. The logarithm narrows that gap to 2.95 against 1.00, and the division by the length turns every count into a small share of the article.

Term frequency looks inside one article only. The cell below counts, for six terms, the number of articles that contain them, and prints the scikit-learn IDF of each.

In [19]:
cv_binary = CountVectorizer(binary=True)
B = cv_binary.fit_transform(df["text"])
doc_freq = np.asarray(B.sum(axis=0)).ravel()
idf_corpus = TfidfVectorizer().fit(df["text"]).idf_
for term in ["the", "said", "year", "profits", "quarterly", "timewarner"]:
    j = cv_binary.vocabulary_[term]
    print(f"{term:11s} DF {doc_freq[j]:5,}   IDF {idf_corpus[j]:.2f}")

the         DF 2,225   IDF 1.00
said        DF 1,888   IDF 1.16
year        DF 1,188   IDF 1.63
profits     DF    81   IDF 4.30
quarterly   DF    20   IDF 5.66
timewarner  DF     1   IDF 8.01


`the` occurs in all **2,225 articles** and keeps the minimum IDF of 1.00, while `timewarner` occurs in a single article and receives **8.01**. The logarithm keeps the range of the weights narrow: a term that is 2,225 times rarer receives a weight only 8 times as large.

## 7. Reading TF-IDF Weights

On a real corpus, the highest TF-IDF weights of an article read as its topic. The cell below builds the TF-IDF matrix with stop words removed and defines `top_terms()`, which returns the terms with the highest values in one row of a matrix.

In [20]:
tv = TfidfVectorizer(stop_words="english")
T = tv.fit_transform(df["text"])
terms = tv.get_feature_names_out()

def top_terms(matrix, row, k=5):
    values = matrix[row].toarray().ravel()
    order = values.argsort()[::-1][:k]
    return [(terms[j], round(float(values[j]), 3)) for j in order]

print(df.loc[0, "title"])
print(top_terms(T, 0))

Ad sales boost Time Warner profit
[('timewarner', 0.511), ('aol', 0.378), ('profits', 0.196), ('warner', 0.166), ('profit', 0.161)]


The first article, about Time Warner's profits, is described by `timewarner`, `aol`, `profits`, `warner` and `profit`.

The cell below compares the highest TF-IDF weights with the highest raw counts for the same article.

In [21]:
cv = CountVectorizer(stop_words="english")
X = cv.fit_transform(df["text"])
counts0 = X[0].toarray().ravel()
print("highest counts:", [(terms[j], int(counts0[j])) for j in counts0.argsort()[::-1][:5]])
print("IDF of 'profits':", round(float(tv.idf_[tv.vocabulary_["profits"]]), 3),
      "| IDF of 'said':", round(float(tv.idf_[tv.vocabulary_["said"]]), 3))

highest counts: [('timewarner', 7), ('aol', 7), ('profits', 5), ('said', 5), ('year', 4)]
IDF of 'profits': 4.301 | IDF of 'said': 1.164


`said` occurs **5 times** in the article, as often as `profits`, but its IDF is 1.164 against 4.301 for `profits`, because `said` appears in 1,888 of the 2,225 articles. The weighting pushes it below the terms that are specific to the article.

The cell below prints the top terms of the first article in each category.

In [22]:
for category in sorted(df["category"].unique()):
    i = df.index[df["category"] == category][0]
    print(f"[{i}] {category:14s} {df.loc[i, 'title']}")
    print("      ", [t for t, _ in top_terms(T, i)])

[0] business       Ad sales boost Time Warner profit
       ['timewarner', 'aol', 'profits', 'warner', 'profit']
[510] entertainment  Gallery unveils interactive tree
       ['tree', 'wentworth', 'sculptor', 'gallery', 'tate']
[896] politics       Labour plans maternity pay rise
       ['maternity', 'pay', 'mothers', 'months', 'hewitt']
[1313] sport          Claxton hunting first major medal
       ['claxton', 'hurdles', 'medal', '60m', 'european']
[1824] tech           Ink helps drive democracy in Asia
       ['ink', 'elections', 'kyrgyz', 'use', 'voter']


In every category, the top terms name the subject of the article: a sculpture at the Tate, maternity pay, an athlete's hurdles season, and the ink used to mark voters in an election.

## 8. N-Gram Features

An **n-gram** is a sequence of n consecutive tokens. Adding bigram columns to the matrix records some of the word order that the bag of words discards.

The cell below builds the bigram columns of three short sentences with `ngram_range=(2, 2)`, which keeps the bigrams only.

In [23]:
sentences = ["bank cuts interest rates again", "bank raises interest rates", "rates fell and interest rose"]
cv2 = CountVectorizer(ngram_range=(2, 2))
X2 = cv2.fit_transform(sentences)
pd.DataFrame(X2.toarray(), index=sentences, columns=cv2.get_feature_names_out())

,and interest,bank cuts,bank raises,cuts interest,fell and,interest rates,interest rose,raises interest,rates again,rates fell
bank cuts interest rates again,0,1,0,1,0,1,0,0,1,0
bank raises interest rates,0,0,1,0,0,1,0,1,0,0
rates fell and interest rose,1,0,0,0,1,0,1,0,0,1


The three sentences produce **10 bigram columns**. `interest rates` occurs in the first two sentences. The third sentence contains both `interest` and `rates` but has no `interest rates` column, because the two words are not adjacent.

An n-gram is formed after preprocessing. The cell below builds the same columns with the scikit-learn English stop list, which contains the word `interest`.

In [24]:
cv2_stop = CountVectorizer(ngram_range=(2, 2), stop_words="english")
cv2_stop.fit(sentences)
print("interest on the stop list:", "interest" in cv2_stop.get_stop_words())
print(cv2_stop.get_feature_names_out())

interest on the stop list: True
['bank cuts' 'bank raises' 'cuts rates' 'fell rose' 'raises rates'
 'rates fell']


`interest` is removed before the bigrams are formed, so `interest rates` never forms, and bigrams such as `cuts rates` appear although those two words are never adjacent in the text. A stop list chosen for single words can break the phrases a bigram is meant to capture.

The cell below measures the cost on the corpus: the number of columns for single terms, for terms and bigrams, and for terms, bigrams and trigrams, with `min_df=1` and with `min_df=2`.

In [25]:
rows = []
for ngram_range, label in (((1, 1), "single terms"), ((1, 2), "terms and bigrams"), ((1, 3), "terms, bigrams and trigrams")):
    for min_df in (1, 2):
        M = CountVectorizer(stop_words="english", ngram_range=ngram_range, min_df=min_df).fit_transform(df["text"])
        rows.append({"features": label, "min_df": min_df, "columns": M.shape[1], "non-zero cells": M.nnz})
pd.DataFrame(rows).set_index(["features", "min_df"])

columns  non-zero cells
features                    min_df                         
single terms                1         28980          319067
                            2         17351          307438
terms and bigrams           1        342635          744323
                            2         76375          478063
terms, bigrams and trigrams 1        728434         1183731
                            2        120455          575752

Bigrams multiply the number of columns by **11.8**, from 28,980 to 342,635. With `min_df=2`, the count falls to 76,375.

The cell below counts the bigrams that occur in a single article.

In [26]:
bigram_matrix = CountVectorizer(stop_words="english", ngram_range=(2, 2), binary=True).fit_transform(df["text"])
once = int((np.asarray(bigram_matrix.sum(axis=0)).ravel() == 1).sum())
print(f"bigrams: {bigram_matrix.shape[1]:,} | in one article only: {once:,} ({once / bigram_matrix.shape[1]:.0%})")

bigrams: 313,655 | in one article only: 254,631 (81%)


**254,631** of the 313,655 bigrams, 81%, occur in one article only. Such a column cannot help a model on a new document, which is why `min_df=2` removes most of them.

## Exercises

### Exercise 1: Binary and Logarithmic Term Frequency

Using the three documents, build the TF-IDF matrix three times:

- With the default settings.

- With `binary=True`, which records whether each term occurs in place of the count.

- With `sublinear_tf=True`, which replaces the count tf by 1 + ln(tf).

Print the row of d3 under each setting and, in a Markdown cell, state which term changes the most and why.

In [27]:
# TODO: Build the TF-IDF matrix with the default settings, with binary=True and with sublinear_tf=True, then print the row of d3 for each.


**Expected results:**

- **Default:** `fell` weighs 0.759 in d3.
- **Binary:** `fell` falls to 0.504, below `and`, because its count of 2 is recorded as 1 and the lower IDF of `fell` then decides.
- **Sublinear:** `fell` weighs 0.703, between the two, because 1 + ln(2) = 1.69 in place of 2.

`fell` changes the most because it is the only term with a count above 1.

### Exercise 2: Vocabulary Settings

Build the count matrix of the corpus with stop words removed and `min_df` set to 1, 2, 5 and 10. For each setting print the vocabulary size and the share of the 447,008 tokens that are still counted.

Then add one run with `min_df=2` and `max_df=0.5`, which also drops every term that occurs in more than 1/2 of the articles.

In a Markdown cell, state which setting halves the vocabulary while keeping at least 90% of the tokens.

In [28]:
# TODO: Build the matrix for min_df in (1, 2, 5, 10) and for min_df=2 with max_df=0.5, and print the vocabulary size and the share of tokens kept.


**Expected result:** `min_df=5` reduces the vocabulary from 28,980 to 9,052 terms, less than 1/3, while keeping 90% of the tokens. `min_df=2` keeps 97% of the tokens with 17,351 terms. `max_df=0.5` removes only 2 further terms, because the stop list has already taken out the words that occur in most articles.

### Exercise 3: Top Terms per Category

Average the TF-IDF rows of every article in each category, so that each category becomes one vector, and print the **10 highest-weighted terms** of each category.

`T[mask].mean(axis=0)` averages the rows selected by a boolean mask, and `np.asarray(...).ravel()` turns the result into a flat array.

In a Markdown cell, state whether the terms read as the category, and name one term that appears in every category's list.

In [29]:
# TODO: Average the TF-IDF rows per category and print the 10 highest-weighted terms of each category.


**Expected result:** The lists read as the categories: `growth`, `economy`, `bank` and `market` for business, `film`, `music` and `awards` for entertainment, `labour`, `election` and `blair` for politics, `england`, `game`, `cup` and `match` for sport, and `mobile`, `technology`, `software` and `users` for tech. The term `said` appears in every list, because it occurs in most articles and is absent from the stop list.

### Exercise 4 (Extension): Stemming Before Counting

`CountVectorizer` and `TfidfVectorizer` accept an `analyzer` argument: a function that receives one document and returns its list of tokens. Write an analyzer that lowercases the text, keeps the tokens of `CountVectorizer`'s default pattern, drops the scikit-learn stop words, and applies the Porter stemmer from NLTK to each token.

Build the TF-IDF matrix with this analyzer and print:

- The vocabulary size, compared with the 28,980 terms of the unstemmed matrix.

- The 5 highest-weighted terms of the first article.

In [30]:
# TODO: Write a stemming analyzer, build the TF-IDF matrix with it, and print the vocabulary size and the top terms of the first article.


**Expected result:** Stemming reduces the vocabulary to **20,361 terms**. The top terms of the first article become `timewarn`, `aol`, `profit`, `warner` and `revenu`: `profits` and `profit` have merged into one column whose weight, about 0.31, exceeds either of the two original weights.